# GSoC 2026 — NeuroDyads Pre-Task
## Brain-to-Brain Decoder: Validating Neural Synchrony in Human Conversation

**Submitted by:** Vishal Shankar
**Organization:** ML4SCI
**Mentors:** Dr. Evie Malaia & Dr. Brendan Ames
**Date:** March 2026

---

This notebook is my complete solution to the NeuroDyads pre-task. The task involves preprocessing two simultaneous EEG recordings from a conversational dyad, learning joint low-dimensional embeddings using CEBRA, and interpreting what the embedding geometry reveals about neural synchrony during conversation.

The data consists of two raw 64-channel EEG files (Speaker and Listener) recorded simultaneously at 250 Hz using EGI HydroCel nets. The recordings contain three DIN1 markers that define two conversational segments — one with positive affect and one with negative affect.

**My pipeline:**
- **Part 1**: Segmentation by DIN1 markers, VREF channel removal, ICA artifact removal, power spectrum comparison
- **Part 2**: CEBRA embedding on the joint 128-channel matrix with shuffled control
- **Part 3**: Interpreting the embedding geometry and what the control tells us
- **Part 4**: Honest reflection on the biggest limitation of my specific analysis


## Part 1: Preprocessing

### Step 1 — Load Raw EDF Files and Inspect Annotations

Before doing anything, I need to see the raw file structure — how many channels, what the sampling rate is, and most importantly, where the DIN1 markers fall. The entire analysis depends on correct segmentation: if the two files are not cropped to the exact same time window, the joint matrix will not be time-locked and CEBRA will learn nothing meaningful.


In [11]:
import mne
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Load both raw EDF files
raw_speaker = mne.io.read_raw_edf('../data/Speaker.edf', preload=True, verbose=False)
raw_listener = mne.io.read_raw_edf('../data/Listener.edf', preload=True, verbose=False)

print("=== SPEAKER ===")
print(f"Channels: {len(raw_speaker.ch_names)}")
print(f"Sampling frequency: {raw_speaker.info['sfreq']} Hz")
print(f"Duration: {raw_speaker.times[-1]:.1f} seconds")
print(f"Annotations:\n{raw_speaker.annotations}")

print("\n=== LISTENER ===")
print(f"Channels: {len(raw_listener.ch_names)}")
print(f"Sampling frequency: {raw_listener.info['sfreq']} Hz")
print(f"Duration: {raw_listener.times[-1]:.1f} seconds")
print(f"Annotations:\n{raw_listener.annotations}")


=== SPEAKER ===
Channels: 65
Sampling frequency: 250.0 Hz
Duration: 303.0 seconds
Annotations:
<Annotations | 5 segments: DIN1 (3), VBeg (1), VEnd (1)>

=== LISTENER ===
Channels: 65
Sampling frequency: 250.0 Hz
Duration: 303.0 seconds
Annotations:
<Annotations | 3 segments: DIN1 (3)>


### Step 2 — Extract DIN1 Marker Times

Both files have 65 channels and exactly 3 DIN1 markers as expected. Duration matches (303 seconds each) which is a good sign that the recordings are aligned. 

Now I need to extract the exact timestamps of the 3 DIN1 markers to perform the segmentation:
- **Segment 1 (positive affect):** DIN1 marker 1 → DIN1 marker 2
- **Segment 2 (negative affect):** DIN1 marker 3 → end of file


In [14]:
# Extract DIN1 marker onset times from Speaker file
# (Both files have the same markers - using Speaker as reference)

din1_onsets = [ann['onset'] for ann in raw_speaker.annotations if ann['description'] == 'DIN1']

print("DIN1 marker times (seconds):")
for i, t in enumerate(din1_onsets):
    print(f"  Marker {i+1}: {t:.3f}s")

tmin_pos = din1_onsets[0]   # positive affect start
tmax_pos = din1_onsets[1]   # positive affect end
tmin_neg = din1_onsets[2]   # negative affect start
tmax_neg = raw_speaker.times[-1]  # negative affect end = end of file

print(f"\nSegment 1 — Positive affect: {tmin_pos:.3f}s → {tmax_pos:.3f}s  (duration: {tmax_pos - tmin_pos:.1f}s)")
print(f"Segment 2 — Negative affect: {tmin_neg:.3f}s → {tmax_neg:.3f}s  (duration: {tmax_neg - tmin_neg:.1f}s)")


DIN1 marker times (seconds):
  Marker 1: 0.788s
  Marker 2: 148.561s
  Marker 3: 148.835s

Segment 1 — Positive affect: 0.788s → 148.561s  (duration: 147.8s)
Segment 2 — Negative affect: 148.835s → 302.996s  (duration: 154.2s)


### Step 3 — Segment and Remove VREF (Channel 65)

The markers are clean:
- **Positive affect:** 0.79s → 148.56s (~147.8 seconds)
- **Negative affect:** 148.84s → 303.0s (~154.2 seconds)

The gap between marker 2 and 3 is only 0.274 seconds — essentially a seamless transition between the two conversational conditions.

Now I will crop both files to these two segments and remove channel 65 (VREF — the vertex reference electrode). VREF is not an EEG signal; it's the hardware reference and must be dropped before ICA, otherwise it will appear as a dominant component and distort the decomposition.


In [17]:
def segment_and_clean(raw, tmin, tmax, label):
    """Crop to segment and drop VREF channel."""
    r = raw.copy().crop(tmin=tmin, tmax=tmax)
    
    # Drop VREF (channel 65 — index 64)
    vref = [ch for ch in r.ch_names if 'VREF' in ch.upper() or ch == 'E65']
    if vref:
        r.drop_channels(vref)
        dropped = vref[0]
    else:
        dropped = r.ch_names[64]
        r.drop_channels([dropped])
    
    print(f"{label}: {len(r.ch_names)} channels, {r.times[-1]:.1f}s  | Dropped: {dropped}")
    return r

# Segment both files into positive and negative affect
spk_pos = segment_and_clean(raw_speaker, tmin_pos, tmax_pos, "Speaker  — Positive")
spk_neg = segment_and_clean(raw_speaker, tmin_neg, tmax_neg, "Speaker  — Negative")
lst_pos = segment_and_clean(raw_listener, tmin_pos, tmax_pos, "Listener — Positive")
lst_neg = segment_and_clean(raw_listener, tmin_neg, tmax_neg, "Listener — Negative")

print("\nAll 4 segments created — 64 channels each.")


Speaker  — Positive: 64 channels, 147.8s  | Dropped: EEG VREF
Speaker  — Negative: 64 channels, 154.2s  | Dropped: EEG VREF
Listener — Positive: 64 channels, 147.8s  | Dropped: EEG VREF
Listener — Negative: 64 channels, 154.2s  | Dropped: EEG VREF

All 4 segments created — 64 channels each.


### Step 4 — ICA Artifact Removal

With the segments properly cropped and VREF removed, I now apply ICA to remove artifacts — primarily eye blinks (EOG) and muscle noise (EMG).

**Why ICA?** ICA decomposes the multichannel EEG into statistically independent components. Artifacts like eye blinks produce a characteristic frontal topography and slow waveform that are easy to identify. Removing them before feeding data to CEBRA ensures the embedding reflects genuine neural synchrony rather than shared movement artifacts between participants.

**My approach:**
- Apply a 1–40 Hz bandpass filter before ICA fitting (ICA is sensitive to slow drifts)
- Use 20 components — enough to capture dominant sources in 64-channel data without overfitting
- Use MNE's automatic EOG detection to flag blink components
- I will inspect and justify each rejected component


In [20]:
from mne.preprocessing import ICA

def run_ica(raw_seg, n_components=20, label=""):
    """Filter, fit ICA, auto-detect EOG components, return clean raw + ICA object."""
    # Bandpass 1-40 Hz for ICA fitting
    raw_filt = raw_seg.copy().filter(1.0, 40.0, verbose=False)
    
    ica = ICA(n_components=n_components, random_state=42, max_iter='auto', verbose=False)
    ica.fit(raw_filt, verbose=False)
    
    # Auto-detect EOG artifact components using frontal channels
    # EGI HydroCel nets: frontal channels are typically E8, E14, E21, E25
    frontal_chs = [ch for ch in raw_seg.ch_names if ch in ['E8','E14','E21','E25','EEG E8','EEG E14']]
    
    if frontal_chs:
        eog_idx, scores = ica.find_bads_eog(raw_filt, ch_name=frontal_chs[0], verbose=False)
    else:
        eog_idx = []
    
    ica.exclude = eog_idx
    
    # Apply ICA to original (unfiltered) segment
    raw_clean = raw_seg.copy()
    ica.apply(raw_clean, verbose=False)
    
    print(f"{label}: excluded components {eog_idx}")
    return raw_clean, ica

print("Running ICA on all 4 segments...")
spk_pos_clean, ica_spk_pos = run_ica(spk_pos, label="Speaker  — Positive")
spk_neg_clean, ica_spk_neg = run_ica(spk_neg, label="Speaker  — Negative")
lst_pos_clean, ica_lst_pos = run_ica(lst_pos, label="Listener — Positive")
lst_neg_clean, ica_lst_neg = run_ica(lst_neg, label="Listener — Negative")

print("\nICA complete.")


Running ICA on all 4 segments...
Speaker  — Positive: excluded components []
Speaker  — Negative: excluded components []
Listener — Positive: excluded components []
Listener — Negative: excluded components []

ICA complete.


In [22]:
# Auto-detection didn't find EOG channels by name — EGI nets use generic naming.
# Instead, we exclude the top variance components which typically capture 
# eye blink and muscle artifacts in short-segment EEG data.
# We inspect the variance explained by each component and exclude
# the top 2 components from each participant as a conservative approach.

def exclude_top_components(ica, raw_seg, n_exclude=2, label=""):
    """Exclude top n components by variance — standard approach when no EOG channel."""
    # Get explained variance per component
    var = ica.get_explained_variance_ratio(raw_seg, components=list(range(ica.n_components_)))
    
    # Sort by variance descending, take top n
    sorted_comps = sorted(var.items(), key=lambda x: x[1], reverse=True)
    top_comps = [c[0] for c in sorted_comps[:n_exclude]]
    
    ica.exclude = top_comps
    raw_clean = raw_seg.copy()
    ica.apply(raw_clean, verbose=False)
    
    print(f"{label}: excluded components {top_comps} "
          f"(variance: {[round(var[c]*100,1) for c in top_comps]}%)")
    return raw_clean

spk_pos_clean = exclude_top_components(ica_spk_pos, spk_pos, label="Speaker  — Positive")
spk_neg_clean = exclude_top_components(ica_spk_neg, spk_neg, label="Speaker  — Negative")
lst_pos_clean = exclude_top_components(ica_lst_pos, lst_pos, label="Listener — Positive")
lst_neg_clean = exclude_top_components(ica_lst_neg, lst_neg, label="Listener — Negative")

print("\nArtifact removal complete.")


Speaker  — Positive: excluded components ['eeg'] (variance: [-33.6]%)
Speaker  — Negative: excluded components ['eeg'] (variance: [82.8]%)
Listener — Positive: excluded components ['eeg'] (variance: [92.6]%)
Listener — Negative: excluded components ['eeg'] (variance: [94.2]%)

Artifact removal complete.


In [24]:
# Fix: Use component index directly — exclude component 0 and 1
# which are conventionally the highest-variance components in ICA
# These capture the dominant artifacts (eye blinks, cardiac, slow drift)

def apply_ica_fixed(ica, raw_seg, exclude_comps, label=""):
    ica.exclude = exclude_comps
    raw_clean = raw_seg.copy()
    ica.apply(raw_clean, verbose=False)
    print(f"{label}: excluded ICA components {exclude_comps}")
    return raw_clean

# Exclude components 0 and 1 — highest variance components
# Rationale: In continuous resting/conversational EEG, the first ICA components
# almost universally capture eye movement (frontal, low frequency) and 
# cardiac artifacts. This is the standard conservative approach.
spk_pos_clean = apply_ica_fixed(ica_spk_pos, spk_pos, [0, 1], "Speaker  — Positive")
spk_neg_clean = apply_ica_fixed(ica_spk_neg, spk_neg, [0, 1], "Speaker  — Negative")
lst_pos_clean = apply_ica_fixed(ica_lst_pos, lst_pos, [0, 1], "Listener — Positive")
lst_neg_clean = apply_ica_fixed(ica_lst_neg, lst_neg, [0, 1], "Listener — Negative")

print("\nICA artifact removal applied — components 0 & 1 excluded from all segments.")


Speaker  — Positive: excluded ICA components [0, 1]
Speaker  — Negative: excluded ICA components [0, 1]
Listener — Positive: excluded ICA components [0, 1]
Listener — Negative: excluded ICA components [0, 1]

ICA artifact removal applied — components 0 & 1 excluded from all segments.


### Step 5 — Power Spectrum: Before vs After ICA (Required Comparison)

I plot the power spectral density (PSD) of the Speaker's positive affect segment before and after ICA on the same axes. This shows what the artifact removal actually changed in the frequency domain.

In [29]:
import os

# Compute PSD before and after ICA for Speaker — Positive segment
psd_before = spk_pos.compute_psd(method='welch', fmin=0.5, fmax=45, verbose=False)
psd_after  = spk_pos_clean.compute_psd(method='welch', fmin=0.5, fmax=45, verbose=False)

# Average across channels
freqs = psd_before.freqs
power_before = psd_before.get_data().mean(axis=0)
power_after  = psd_after.get_data().mean(axis=0)

# Convert to dB
power_before_db = 10 * np.log10(power_before)
power_after_db  = 10 * np.log10(power_after)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(freqs, power_before_db, color='steelblue', linewidth=1.5, label='Before ICA')
ax.plot(freqs, power_after_db,  color='tomato',    linewidth=1.5, label='After ICA')
ax.set_xlabel('Frequency (Hz)', fontsize=12)
ax.set_ylabel('Power (dB)', fontsize=12)
ax.set_title('Speaker — Positive Affect\nPower Spectral Density: Before vs After ICA', fontsize=13)
ax.legend(fontsize=11)
ax.axvspan(0.5, 4, alpha=0.08, color='purple', label='Delta')
ax.axvspan(8, 12, alpha=0.08, color='green',  label='Alpha')
ax.grid(True, alpha=0.3)
plt.tight_layout()

os.makedirs('../figures', exist_ok=True)
plt.savefig('../figures/psd_before_after_ica.png', dpi=150)
plt.show()
print("Figure saved to ../figures/psd_before_after_ica.png")


Figure saved to ../figures/psd_before_after_ica.png


### PSD Interpretation

The power spectrum before and after ICA shows the effect of removing ICA components 0 and 1.

**What I observe:**
- The low-frequency region (delta, 0.5–4 Hz) shows a reduction in power after ICA. This is consistent with removing slow eye-movement artifacts, which typically manifest as high-amplitude, low-frequency deflections predominantly in frontal channels.
- The alpha band (8–12 Hz) is largely preserved, which is a good sign — it means we did not remove genuine neural oscillations.
- The 1/f slope (power decreasing with frequency) is maintained in both curves, indicating the ICA did not distort the broadband spectral structure.

**Why I excluded components 0 and 1:**  
Without a dedicated EOG channel in this EGI dataset, I could not use MNE's automatic EOG correlator. In 64-channel continuous EEG, ICA components are sorted by variance explained. The first two components nearly always capture the dominant non-neural sources — eye blinks and slow drift — due to their large amplitude relative to neural signals. This is a well-established heuristic in EEG preprocessing literature and a conservative choice: it removes the most contaminated components while keeping the majority of neural signal intact.

**Uncertainty:** I cannot confirm without visual inspection of the component topographies that components 0 and 1 are purely artifactual. With more time I would plot the ICA topographies and time courses to validate this decision.
